# Customer Churn Prediction with scikit-learn

Binary classification project using scikit-learn's `LogisticRegression` inside a full `Pipeline`.

Covers EDA, preprocessing, class-imbalance handling with `class_weight='balanced'`, threshold optimisation via the precision-recall curve, cross-validation, and full model evaluation.

## 1. Setup

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path('..').resolve()
sys.path.insert(0, str(PROJECT_ROOT / 'src'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.metrics import classification_report, roc_auc_score

from churn_sklearn import (
    load_data,
    plot_eda,
    build_pipeline,
    find_best_threshold,
    assign_risk_band,
    evaluate_model,
    FEATURES,
    TARGET,
)

sns.set_theme(style='whitegrid', context='notebook')

## 2. Load Dataset

In [ ]:
df = load_data()
df.head()

## 3. Data Overview

In [ ]:
print(f'Rows: {len(df):,}')
print(f'Churn rate: {df[TARGET].mean():.2%}')
display(df.describe())
display(df.isna().sum())

## 4. Exploratory Data Analysis

In [ ]:
plot_eda(df)

## 5. Train / Test Split

We use `stratify=y` to ensure the churn ratio is preserved in both splits — important with a ~10% minority class.

In [ ]:
X = df[FEATURES]
y = df[TARGET].to_numpy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f'Train: {len(X_train):,} rows  |  Churn rate: {y_train.mean():.2%}')
print(f'Test:  {len(X_test):,} rows  |  Churn rate: {y_test.mean():.2%}')

## 6. Build and Train Pipeline

The `Pipeline` chains `StandardScaler` → `LogisticRegression` so scaling is always fit on training data only, preventing data leakage.

`class_weight='balanced'` automatically upweights the minority class (churned customers) so the model doesn't simply predict "stayed" for everyone.

In [ ]:
pipeline = build_pipeline(class_weight='balanced')
pipeline.fit(X_train, y_train)
print(pipeline)

## 7. Cross-Validation

`StratifiedKFold` preserves the class ratio in every fold. We score by ROC-AUC, which is threshold-independent and well-suited for imbalanced problems.

In [ ]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
cv_auc = cross_val_score(pipeline, X_train, y_train, cv=cv, scoring='roc_auc')

print(f'5-Fold CV ROC-AUC scores: {cv_auc.round(3)}')
print(f'Mean: {cv_auc.mean():.3f}  ±  {cv_auc.std():.3f}')

## 8. Threshold Optimisation

Even with `class_weight='balanced'`, the default 0.5 threshold may not be optimal. We use the precision-recall curve to find the threshold that maximises F1-score on the training set.

In [ ]:
y_prob_train = pipeline.predict_proba(X_train)[:, 1]
best_threshold = find_best_threshold(y_train, y_prob_train)
print(f'Optimised threshold: {best_threshold:.2f}')

# Compare default vs optimised threshold on test set
for label, thresh in [('Default (0.5)', 0.5), (f'Optimised ({best_threshold:.2f})', best_threshold)]:
    y_pred_t = (pipeline.predict_proba(X_test)[:, 1] >= thresh).astype(int)
    print(f'\n--- {label} ---')
    print(classification_report(y_test, y_pred_t, target_names=['Stayed', 'Churned']))

## 9. Full Evaluation on Test Set

In [ ]:
results = evaluate_model(pipeline, X_test, y_test, threshold=best_threshold)
print(f'ROC-AUC: {results["auc"]:.3f}')
print(results['classification_report'])

## 10. Feature Importance

In [ ]:
display(results['feature_importance'])

## 11. Churn Risk Bands

In [ ]:
test_results = pd.DataFrame({
    'actual_churn': y_test,
    'predicted_churn_probability': results['y_prob'],
    'predicted_churn': results['y_pred'],
})
test_results['risk_band'] = test_results['predicted_churn_probability'].apply(assign_risk_band)

display(test_results.head(10))
display(test_results['risk_band'].value_counts())

## 12. Key Takeaways

- **Class imbalance** (~10% churn) is handled by `class_weight='balanced'` and stratified splitting
- **Pipeline** ensures the scaler is always fit on training data only — no data leakage
- **Cross-validation** gives a reliable estimate of generalisation performance
- **Precision-recall threshold** is optimised for F1 rather than defaulting to 0.5
- **Risk bands** translate probabilities into actionable business categories